# Lexel v2 Stage 1: QLoRA SFT for `Lexel 0.5B (Lite)` (Kaggle T4 / Unsloth)

Trains the Stage 1 Supervised Fine-Tuning (SFT) adapter for **Lexel 0.5B (Lite)** (`unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit`) using Unsloth 4-bit QLoRA with automatic HuggingFace PEFT/TRL fallback.

## Research-Grounded Defaults (`ml/RESEARCH_NOTES.md`)
- **Tier**: `lexel-0.5b` (`r=16`, `lora_alpha=32`, all 7 linear projections, `max_seq_length=1536`)
- **SFT Schedule**: `lr=0.0003`, `2.0` epochs (empirical Prose Session 2 ceiling), effective batch size `16`, cosine decay with `5%` warmup.
- **Completion-Only Loss Masking**: Applies `train_on_responses_only` and decodes a real collated batch before `.train()` to verify `supervised_count > 0` and zero prompt tokens in labels.
- **Tier Override**: Set `LEXEL_MODEL_TIER` to `lexel-1.5b`, `lexel-0.5b`, or `qwen3-1.7b-bakeoff` to switch configs without editing cells.

In [ ]:
# Cell 1: Environment Setup & Kaggle Package Installation (Unsloth imported BEFORE torch/trl)
import importlib.util
import os
import subprocess
import sys

os.environ["WANDB_DISABLED"] = "true"
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Pin GPU 0 on Kaggle 2x T4 so 4-bit bitsandbytes QLoRA avoids DataParallel device-mismatch errors
if "KAGGLE_KERNEL_RUN_TYPE" in os.environ and "CUDA_VISIBLE_DEVICES" not in os.environ:
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"

def _pkg_installed(name: str) -> bool:
    if name in sys.modules:
        return True
    try:
        return importlib.util.find_spec(name) is not None
    except (ValueError, ImportError):
        return False

def ensure_training_packages() -> None:
    required = ("trl", "peft", "datasets", "accelerate", "bitsandbytes")
    missing = [pkg for pkg in required if not _pkg_installed(pkg)]
    has_unsloth = _pkg_installed("unsloth")

    if "KAGGLE_KERNEL_RUN_TYPE" in os.environ and (missing or not has_unsloth):
        print("Installing Unsloth + QLoRA training dependencies for Kaggle GPU...")
        try:
            subprocess.check_call(
                [
                    sys.executable,
                    "-m",
                    "pip",
                    "install",
                    "-q",
                    "--upgrade",
                    "unsloth",
                    "unsloth_zoo",
                    "trl",
                    "peft",
                    "accelerate",
                    "bitsandbytes",
                    "datasets",
                ]
            )
        except Exception as pip_exc:
            print(f"WARNING: pip install failed ({pip_exc}). Enable Internet in Kaggle Settings.")
    elif missing:
        print(f"Installing missing packages: {missing}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

ensure_training_packages()

# Import unsloth BEFORE torch/transformers/trl so fused kernels patch cleanly
try:
    import unsloth
    from unsloth import FastLanguageModel, is_bfloat16_supported
    from unsloth.chat_templates import get_chat_template, train_on_responses_only
    HAS_UNSLOTH = True
except Exception as exc:
    print(f"Unsloth fast path unavailable ({exc}); using standard Transformers + PEFT + TRL fallback.")
    HAS_UNSLOTH = False

import torch
if not HAS_UNSLOTH:
    def is_bfloat16_supported() -> bool:
        return torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8

print(f"PyTorch: {torch.__version__} | CUDA available: {torch.cuda.is_available()} | Unsloth active: {HAS_UNSLOTH}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)} | BF16: {is_bfloat16_supported()}")

In [ ]:
# Cell 2: Per-Tier Training Config & Automatic Dataset Discovery
import hashlib
import inspect
import json
import time
from datetime import UTC, datetime
from pathlib import Path

from datasets import load_dataset
from trl import SFTConfig, SFTTrainer

TIER_CONFIGS = json.loads(r'''{
    "lexel-0.5b": {
        "tier_id": "lexel-0.5b",
        "label": "Lexel 0.5B (Lite)",
        "tier_kind": "lite",
        "unsloth_model_id": "unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit",
        "hf_model_id": "Qwen/Qwen2.5-0.5B-Instruct",
        "chat_template": "qwen-2.5",
        "enable_thinking": false,
        "instruction_part": "<|im_start|>user\n",
        "response_part": "<|im_start|>assistant\n",
        "max_seq_length": 1536,
        "lora_r": 16,
        "lora_alpha": 32,
        "lora_dropout": 0.0,
        "target_modules": [
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
            "gate_proj",
            "up_proj",
            "down_proj"
        ],
        "sft_learning_rate": 0.0003,
        "sft_epochs": 2.0,
        "sft_per_device_batch_size": 2,
        "sft_grad_accum_steps": 8,
        "sft_warmup_ratio": 0.05,
        "sft_weight_decay": 0.01,
        "sft_eval_steps": 50,
        "sft_save_steps": 100,
        "simpo_beta": 2.0,
        "simpo_gamma": 0.8,
        "simpo_unmasked_token_weight": 0.05,
        "simpo_sft_weight": 0.05,
        "simpo_learning_rate": 8e-06,
        "simpo_epochs": 1.5,
        "simpo_per_device_batch_size": 2,
        "simpo_grad_accum_steps": 8,
        "simpo_warmup_ratio": 0.1,
        "simpo_weight_decay": 0.01,
        "quant_methods": [
            "q4_k_m",
            "q5_k_m"
        ],
        "gguf_filename_prefix": "qwen2.5-0.5b-instruct",
        "ollama_tag": "qwen2.5:0.5b",
        "approx_ram_mb_q4": 450,
        "approx_ram_mb_q5": 520,
        "seed": 42,
        "effective_sft_batch_size": 16,
        "effective_simpo_batch_size": 16
    },
    "lexel-1.5b": {
        "tier_id": "lexel-1.5b",
        "label": "Lexel 1.5B (Standard)",
        "tier_kind": "standard",
        "unsloth_model_id": "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit",
        "hf_model_id": "Qwen/Qwen2.5-1.5B-Instruct",
        "chat_template": "qwen-2.5",
        "enable_thinking": false,
        "instruction_part": "<|im_start|>user\n",
        "response_part": "<|im_start|>assistant\n",
        "max_seq_length": 1536,
        "lora_r": 32,
        "lora_alpha": 64,
        "lora_dropout": 0.0,
        "target_modules": [
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
            "gate_proj",
            "up_proj",
            "down_proj"
        ],
        "sft_learning_rate": 0.0002,
        "sft_epochs": 2.0,
        "sft_per_device_batch_size": 2,
        "sft_grad_accum_steps": 8,
        "sft_warmup_ratio": 0.05,
        "sft_weight_decay": 0.01,
        "sft_eval_steps": 50,
        "sft_save_steps": 100,
        "simpo_beta": 2.0,
        "simpo_gamma": 1.0,
        "simpo_unmasked_token_weight": 0.05,
        "simpo_sft_weight": 0.05,
        "simpo_learning_rate": 5e-06,
        "simpo_epochs": 1.5,
        "simpo_per_device_batch_size": 2,
        "simpo_grad_accum_steps": 8,
        "simpo_warmup_ratio": 0.1,
        "simpo_weight_decay": 0.01,
        "quant_methods": [
            "q4_k_m",
            "q5_k_m"
        ],
        "gguf_filename_prefix": "qwen2.5-1.5b-instruct",
        "ollama_tag": "qwen2.5:1.5b",
        "approx_ram_mb_q4": 950,
        "approx_ram_mb_q5": 1150,
        "seed": 42,
        "effective_sft_batch_size": 16,
        "effective_simpo_batch_size": 16
    },
    "qwen3-1.7b-bakeoff": {
        "tier_id": "qwen3-1.7b-bakeoff",
        "label": "Qwen3 1.7B Non-Thinking (Bake-Off)",
        "tier_kind": "bakeoff",
        "unsloth_model_id": "unsloth/Qwen3-1.7B-unsloth-bnb-4bit",
        "hf_model_id": "Qwen/Qwen3-1.7B",
        "chat_template": "qwen-3",
        "enable_thinking": false,
        "instruction_part": "<|im_start|>user\n",
        "response_part": "<|im_start|>assistant\n",
        "max_seq_length": 1536,
        "lora_r": 32,
        "lora_alpha": 64,
        "lora_dropout": 0.0,
        "target_modules": [
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
            "gate_proj",
            "up_proj",
            "down_proj"
        ],
        "sft_learning_rate": 0.00015,
        "sft_epochs": 2.0,
        "sft_per_device_batch_size": 2,
        "sft_grad_accum_steps": 8,
        "sft_warmup_ratio": 0.05,
        "sft_weight_decay": 0.01,
        "sft_eval_steps": 50,
        "sft_save_steps": 100,
        "simpo_beta": 2.0,
        "simpo_gamma": 1.0,
        "simpo_unmasked_token_weight": 0.05,
        "simpo_sft_weight": 0.05,
        "simpo_learning_rate": 4e-06,
        "simpo_epochs": 1.5,
        "simpo_per_device_batch_size": 2,
        "simpo_grad_accum_steps": 8,
        "simpo_warmup_ratio": 0.1,
        "simpo_weight_decay": 0.01,
        "quant_methods": [
            "q4_k_m",
            "q5_k_m"
        ],
        "gguf_filename_prefix": "qwen3-1.7b-nonthinking",
        "ollama_tag": "qwen3:1.7b",
        "approx_ram_mb_q4": 1150,
        "approx_ram_mb_q5": 1400,
        "seed": 42,
        "effective_sft_batch_size": 16,
        "effective_simpo_batch_size": 16
    }
}''')

DEFAULT_TIER = 'lexel-0.5b'
MODEL_TIER = os.environ.get("LEXEL_MODEL_TIER", DEFAULT_TIER)
if MODEL_TIER not in TIER_CONFIGS:
    raise ValueError(f"Unsupported LEXEL_MODEL_TIER={MODEL_TIER!r}. Valid: {sorted(TIER_CONFIGS)}")

CFG = TIER_CONFIGS[MODEL_TIER]
BASE_MODEL_ID = CFG["unsloth_model_id"] if HAS_UNSLOTH else CFG["hf_model_id"]
MAX_SEQ_LENGTH = int(CFG["max_seq_length"])
LORA_R = int(CFG["lora_r"])
LORA_ALPHA = int(CFG["lora_alpha"])
LORA_DROPOUT = float(CFG["lora_dropout"])
TARGET_MODULES = list(CFG["target_modules"])

def resolve_data_dir() -> Path:
    candidates = [
        Path(os.environ["LEXEL_DATA_DIR"]) if "LEXEL_DATA_DIR" in os.environ else None,
        Path("/kaggle/input/lexel-crucible-datasets"),
        Path("/kaggle/input/datasets/lawandsoran/lexel-crucible-datasets"),
        Path.cwd() / "ml" / "datasets",
        Path.cwd().parent / "datasets",
        Path.cwd() / "datasets",
    ]
    for cand in candidates:
        if cand and (cand / "lexel_sft_train.jsonl").is_file():
            return cand.resolve()

    kaggle_input = Path("/kaggle/input")
    if kaggle_input.is_dir():
        matches = sorted(kaggle_input.rglob("lexel_sft_train.jsonl"))
        if matches:
            return matches[0].parent.resolve()

    for root_cand in (Path.cwd(), *Path.cwd().parents):
        if (root_cand / "ml" / "crucible" / "dataset_builder.py").is_file():
            if str(root_cand) not in sys.path:
                sys.path.insert(0, str(root_cand))
            from ml.crucible.dataset_builder import build_all_datasets
            out_dir = root_cand / "ml" / "datasets"
            build_all_datasets(out_dir)
            return out_dir.resolve()

    raise FileNotFoundError(
        "Could not find lexel_sft_train.jsonl. Attach the `lexel-crucible-datasets` Kaggle Dataset "
        "(or set LEXEL_DATA_DIR to your ml/datasets directory)."
    )

DATA_DIR = resolve_data_dir()
TRAIN_FILE = DATA_DIR / "lexel_sft_train.jsonl"
VAL_FILE = DATA_DIR / "lexel_sft_val.jsonl"
OUTPUT_DIR = Path(os.environ.get("LEXEL_SFT_OUT", f"./outputs/{MODEL_TIER}-sft-adapter")).resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Selected Tier: {MODEL_TIER} ({CFG['label']}) -> {BASE_MODEL_ID}")
print(f"LoRA: r={LORA_R}, alpha={LORA_ALPHA}, lr={CFG['sft_learning_rate']}, epochs={CFG['sft_epochs']}")
print(f"Train file: {TRAIN_FILE} ({TRAIN_FILE.stat().st_size:,} bytes)")
print(f"Val file:   {VAL_FILE} ({VAL_FILE.stat().st_size:,} bytes)")
print(f"Output dir: {OUTPUT_DIR}")

In [ ]:
# Cell 3: Load 4-bit Base Model & Attach All-Linear LoRA Adapters
if HAS_UNSLOTH:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL_ID,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=True,
    )
    tokenizer = get_chat_template(
        tokenizer,
        chat_template=CFG["chat_template"],
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_R,
        target_modules=TARGET_MODULES,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=int(CFG["seed"]),
        use_rslora=False,
        loftq_config=None,
    )
else:
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16 if is_bfloat16_supported() else torch.float16,
    )
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        quantization_config=bnb_config if torch.cuda.is_available() else None,
        device_map="auto" if torch.cuda.is_available() else None,
    )
    if torch.cuda.is_available():
        model = prepare_model_for_kbit_training(model)
    peft_cfg = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=TARGET_MODULES,
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, peft_cfg)

tokenizer.padding_side = "right"
model.print_trainable_parameters()

In [ ]:
# Cell 4: Load Crucible-Verified Datasets & Render Chat Template
dataset = load_dataset(
    "json",
    data_files={
        "train": str(TRAIN_FILE),
        "validation": str(VAL_FILE),
    },
)

apply_sig = inspect.signature(tokenizer.apply_chat_template).parameters

def _render_messages(msgs):
    kwargs = {"tokenize": False, "add_generation_prompt": False}
    if "enable_thinking" in apply_sig:
        kwargs["enable_thinking"] = bool(CFG["enable_thinking"])
    return tokenizer.apply_chat_template(msgs, **kwargs)

def format_chat_prompts(batch):
    return {"text": [_render_messages(msgs) for msgs in batch["messages"]]}

dataset = dataset.map(
    format_chat_prompts,
    batched=True,
    remove_columns=[c for c in dataset["train"].column_names if c != "text"],
)
print(f"Train rows: {len(dataset['train']):,}, Validation rows: {len(dataset['validation']):,}")
print("\nSample rendered ChatML prompt:\n", dataset["train"][0]["text"][:600])

In [ ]:
# Cell 5: Configure SFTTrainer (Version-Safe) & Audit Completion-Only Masking
sft_cfg_params = inspect.signature(SFTConfig.__init__).parameters
sft_trainer_params = inspect.signature(SFTTrainer.__init__).parameters

cfg_kwargs = {
    "output_dir": str(OUTPUT_DIR),
    "per_device_train_batch_size": int(CFG["sft_per_device_batch_size"]),
    "per_device_eval_batch_size": int(CFG["sft_per_device_batch_size"]),
    "gradient_accumulation_steps": int(CFG["sft_grad_accum_steps"]),
    "learning_rate": float(CFG["sft_learning_rate"]),
    "lr_scheduler_type": "cosine",
    "warmup_ratio": float(CFG["sft_warmup_ratio"]),
    "num_train_epochs": float(CFG["sft_epochs"]),
    "optim": "adamw_8bit" if torch.cuda.is_available() else "adamw_torch",
    "weight_decay": float(CFG["sft_weight_decay"]),
    "fp16": bool(torch.cuda.is_available() and not is_bfloat16_supported()),
    "bf16": bool(torch.cuda.is_available() and is_bfloat16_supported()),
    "logging_steps": 10,
    "save_strategy": "steps",
    "save_steps": int(CFG["sft_save_steps"]),
    "save_total_limit": 2,
    "seed": int(CFG["seed"]),
    "report_to": "none",
}

if "eval_strategy" in sft_cfg_params:
    cfg_kwargs["eval_strategy"] = "steps"
    cfg_kwargs["eval_steps"] = int(CFG["sft_eval_steps"])
elif "evaluation_strategy" in sft_cfg_params:
    cfg_kwargs["evaluation_strategy"] = "steps"
    cfg_kwargs["eval_steps"] = int(CFG["sft_eval_steps"])

if "dataset_text_field" in sft_cfg_params:
    cfg_kwargs["dataset_text_field"] = "text"
if "max_seq_length" in sft_cfg_params:
    cfg_kwargs["max_seq_length"] = MAX_SEQ_LENGTH
elif "max_length" in sft_cfg_params:
    cfg_kwargs["max_length"] = MAX_SEQ_LENGTH

# Initialize with packing=False so train_on_responses_only maps exact turn boundaries
if "packing" in sft_cfg_params:
    cfg_kwargs["packing"] = False
if "padding_free" in sft_cfg_params:
    cfg_kwargs["padding_free"] = False

sft_config = SFTConfig(**cfg_kwargs)

trainer_kwargs = {
    "model": model,
    "train_dataset": dataset["train"],
    "eval_dataset": dataset["validation"],
    "args": sft_config,
}
if "processing_class" in sft_trainer_params:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in sft_trainer_params:
    trainer_kwargs["tokenizer"] = tokenizer
if "dataset_text_field" in sft_trainer_params and "dataset_text_field" not in cfg_kwargs:
    trainer_kwargs["dataset_text_field"] = "text"
if "max_seq_length" in sft_trainer_params and "max_seq_length" not in cfg_kwargs and "max_length" not in cfg_kwargs:
    trainer_kwargs["max_seq_length"] = MAX_SEQ_LENGTH

trainer = SFTTrainer(**trainer_kwargs)

# Apply strict completion-only loss masking (-100 on all system/user prompt tokens)
if HAS_UNSLOTH:
    try:
        trainer = train_on_responses_only(
            trainer,
            instruction_part=CFG["instruction_part"],
            response_part=CFG["response_part"],
        )
        print("Applied Unsloth train_on_responses_only completion mask.")
    except Exception as mask_exc:
        print(f"Note: train_on_responses_only fallback ({mask_exc}); continuing with standard SFT collator.")

# Audit one real collated batch before training (per unsloth & dataset-curation skills)
sample_batch = next(iter(trainer.get_train_dataloader()))
supervised_count = int((sample_batch["labels"] != -100).sum().item())
assert supervised_count > 0, "CRITICAL: Zero supervised tokens in batch labels!"
print(f"Batch label audit PASSED: {supervised_count} supervised completion tokens in first batch (shape={tuple(sample_batch['input_ids'].shape)}).")

In [ ]:
# Cell 6: Run Stage 1 SFT Training & Save Adapter + run_manifest.json
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

t0 = time.perf_counter()
train_stats = trainer.train()
elapsed_sec = time.perf_counter() - t0

peak_vram_gb = (
    round(torch.cuda.max_memory_allocated() / (1024**3), 3)
    if torch.cuda.is_available()
    else 0.0
)

adapter_dir = OUTPUT_DIR / "adapter"
adapter_dir.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))

train_sha256 = hashlib.sha256(TRAIN_FILE.read_bytes()).hexdigest()
val_sha256 = hashlib.sha256(VAL_FILE.read_bytes()).hexdigest()

run_manifest = {
    "stage": "stage1_sft",
    "created_at": datetime.now(UTC).isoformat(timespec="seconds"),
    "model_tier": MODEL_TIER,
    "base_model_id": BASE_MODEL_ID,
    "chat_template": CFG["chat_template"],
    "enable_thinking": bool(CFG["enable_thinking"]),
    "max_seq_length": MAX_SEQ_LENGTH,
    "lora": {
        "r": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "lora_dropout": LORA_DROPOUT,
        "target_modules": TARGET_MODULES,
        "use_gradient_checkpointing": "unsloth" if HAS_UNSLOTH else True,
        "adapter_path": str(adapter_dir),
    },
    "hyperparameters": {
        "learning_rate": float(CFG["sft_learning_rate"]),
        "lr_scheduler_type": "cosine",
        "warmup_ratio": float(CFG["sft_warmup_ratio"]),
        "num_train_epochs": float(CFG["sft_epochs"]),
        "effective_batch_size": int(CFG["effective_sft_batch_size"]),
        "optim": cfg_kwargs["optim"],
        "train_on_responses_only": True,
    },
    "datasets": {
        "train_file": str(TRAIN_FILE),
        "train_rows": len(dataset["train"]),
        "train_sha256": train_sha256,
        "val_file": str(VAL_FILE),
        "val_rows": len(dataset["validation"]),
        "val_sha256": val_sha256,
    },
    "telemetry": {
        "global_step": int(trainer.state.global_step),
        "training_loss": round(float(train_stats.training_loss), 6),
        "wall_time_sec": round(elapsed_sec, 2),
        "peak_vram_gb": peak_vram_gb,
    },
}

manifest_path = OUTPUT_DIR / "run_manifest.json"
manifest_path.write_text(json.dumps(run_manifest, indent=2) + "\n", encoding="utf-8")
print("Saved Stage 1 SFT adapter and run_manifest.json to:", OUTPUT_DIR)
print(json.dumps(run_manifest["telemetry"], indent=2))